# Iconographic, Textual, and Bioarchaeological Evidence of Circumcision Practices in Egypt, ca. 3000 BCE–Late Period Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.9jj9-q2tr/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print(f"{metadata['name']}: {metadata['description']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

**Note:** In Croissant datasets, each entity (record set, field, column) is referenced by its `@id`.

In [ ]:
# List record set @ids
record_sets = dataset.metadata.record_sets
if not record_sets:
    print("No record sets present in the metadata. Please check the schema or data sources.")
else:
    print("Record Sets (@id):")
    for rs in record_sets:
        print(f"  - {rs['@id']}: {rs.get('name','(No name)')}")
    # For each record set, list the fields
    for rs in record_sets:
        fields = rs['fields'] if 'fields' in rs else []
        print(f"\nRecord Set '{rs['@id']}' fields:")
        for field in fields:
            print(f"    - Field @id: {field['@id']} ({field.get('name','')})  [Type: {field.get('dataType','')}]")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

**Example:** If the dataset includes a record set for archaeological evidence (e.g., `cr:RecordSet/archaeological-evidence`), reference by its `@id`.

In [ ]:
# Prepare list of record set @ids
record_sets_ids = [rs['@id'] for rs in dataset.metadata.record_sets] if dataset.metadata.record_sets else []
dataframes = {}

for record_set_id in record_sets_ids:
    print(f"Loading records from record set {record_set_id}...")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Columns for record set {record_set_id}: {dataframes[record_set_id].columns.tolist()}")
        print(dataframes[record_set_id].head())
    else:
        print(f"No records found for record set {record_set_id}.")

# Choose a record set for further processing
if dataframes:
    selected_record_set_id = list(dataframes.keys())[0]
    print(f"\nSelected record set for further analysis: {selected_record_set_id}")
    print(f"Fields: {dataframes[selected_record_set_id].columns.tolist()}")
    df = dataframes[selected_record_set_id]
else:
    df = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

**Instructions:**
- Select a numeric field with a known `@id` (from Step 2 or Step 3)
- Filter records with values above a threshold
- Normalize the numeric field
- Optionally group by a categorical field (also referenced via its `@id`)

**Note**: Replace the sample `@id`s with actual ones present in your dataset, e.g., `'cr:Field/age'` or `'cr:Field/period'`.

In [ ]:
if df is not None:
    # Example: Select a numeric field (e.g. 'cr:Field/counts')
    numeric_field_id = None
    for col in df.columns:
        # Try to pick a likely numeric field
        if 'count' in col.lower() or 'number' in col.lower() or 'age' in col.lower():
            numeric_field_id = col
            break
    if not numeric_field_id:
        numeric_field_id = df.select_dtypes(include='number').columns[0] if len(df.select_dtypes(include='number').columns) > 0 else df.columns[0]
    print(f"Using numeric field '{numeric_field_id}' for EDA.")

    threshold = 10
    # Filtering based on threshold
    filtered_df = df[df[numeric_field_id] > threshold] if numeric_field_id in df.columns else df
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalization
    if numeric_field_id in filtered_df.columns and pd.api.types.is_numeric_dtype(filtered_df[numeric_field_id]):
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Grouping by a field
    group_field_id = None
    for col in df.columns:
        if 'period' in col.lower() or 'group' in col.lower() or 'type' in col.lower():
            group_field_id = col
            break
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by '{group_field_id}':")
        print(grouped_df.head())
else:
    print("No DataFrame loaded for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

**Example Plot**: Show numeric field distribution or value by period/group.

**Note:** Please update the code with specific `@id`s as appropriate.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_field_id:
    plt.figure(figsize=(8,5))
    sns.histplot(data=df, x=numeric_field_id, bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()
    
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10,6))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The `mlcroissant` library enables easy loading and exploration of datasets defined with Croissant schemas via URL.
- All data entities such as record sets and fields are referenced using their unique `@id` values.
- This dataset compiles evidence of circumcision practices in Egypt across more than four millennia, with standardized annotation and provenance.
- Exploratory analysis can investigate numeric and categorical patterns, such as counts by period or iconographic frequency.
- Interoperable and reproducible research is supported through FAIR principles and schema-based metadata.

*For further analysis, experiment with other record sets and field `@id`s as documented in the Croissant metadata.*